---
date: 2026-03-17
purpose: >
  Đánh giá khả năng phân loại tâm lý (mental health sentiment) của Google Gemini 2.5
  theo chiến lược zero-shot và few-shot prompting, thay thế hoàn toàn OpenAI.
objectives:
  - Chạy inference với Gemini 2.5 Flash (mặc định) hoặc Pro trên test split
  - So sánh zero-shot vs few-shot về accuracy / macro-F1
  - Lưu đầy đủ artifact: predictions JSONL, metrics JSON, cost report JSON,
    confusion matrix PNG
  - Đảm bảo schema metrics tương thích với M6 comparison_report
---

# M5 — LLM Prompting Evaluation (Gemini 2.5 Flash / Pro)

Notebook này chạy pipeline suy luận LLM cho tập test mental-health và tính các
chỉ số đánh giá, cost report. Kết quả được lưu vào `data/artifacts/` để dùng trong
bước so sánh tổng hợp M6.

> **Yêu cầu**: `GOOGLE_API_KEY` phải được đặt trong file `.env` hoặc biến môi trường.

In [ ]:
"""
Phần 1: Import thư viện và nạp biến môi trường
"""
from __future__ import annotations

import json
import logging
import os
import random
import sys
from pathlib import Path

import pandas as pd
import yaml
from dotenv import load_dotenv
from tqdm.notebook import tqdm

# Thêm thư mục gốc dự án vào sys.path
PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

# Nạp GOOGLE_API_KEY từ file .env
load_dotenv(PROJECT_ROOT / ".env")

from src.models.llm_client import LLMClient, _FEW_SHOT_EXAMPLES
from src.utils.metrics import compute_metrics, save_confusion_matrix_plot, save_metrics

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger(__name__)

print("✓ Thư viện đã nạp xong.")
print(f"  Project root : {PROJECT_ROOT}")
print(f"  GOOGLE_API_KEY : {'✓ Đặt rồi' if os.environ.get('GOOGLE_API_KEY') else '✗ CHƯA đặt — hãy thêm vào .env'}")

In [ ]:
# ---------------------------------------------------------------
# Phần 2: Nạp config và chỉnh tham số thực nghiệm
# ---------------------------------------------------------------
CONFIG_PATH = PROJECT_ROOT / "configs" / "llm_prompting.yaml"

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

# ── Có thể thay đổi ở đây ──────────────────────────────────────
# Model: "gemini-2.5-flash" (nhanh, rẻ) hoặc "gemini-2.5-pro" (chất lượng cao)
MODEL_NAME   = cfg["generation"]["model"]        # mặc định gemini-2.5-flash
PROMPTING_MODE = cfg["prompting"]["mode"]        # "zero_shot" | "few_shot"
SAMPLE_SIZE  = cfg["data"].get("sample_size", 200)   # None = dùng toàn bộ test set
# ────────────────────────────────────────────────────────────────

label_map: dict[int, str] = {int(k): v for k, v in cfg["label_map"].items()}
artifacts_dir = PROJECT_ROOT / cfg["output"]["artifacts_dir"]
artifacts_dir.mkdir(parents=True, exist_ok=True)

print(f"Model          : {MODEL_NAME}")
print(f"Prompting mode : {PROMPTING_MODE}")
print(f"Sample size    : {SAMPLE_SIZE if SAMPLE_SIZE else 'full test set'}")
print(f"Artifacts dir  : {artifacts_dir}")

In [ ]:
# ---------------------------------------------------------------
# Phần 3: Nạp dữ liệu test
# ---------------------------------------------------------------
test_path = PROJECT_ROOT / cfg["data"]["test_path"]
df_test = pd.read_csv(test_path)

if SAMPLE_SIZE and SAMPLE_SIZE < len(df_test):
    rng = random.Random(cfg["data"].get("sample_seed", 42))
    indices = rng.sample(range(len(df_test)), SAMPLE_SIZE)
    df_test = df_test.iloc[indices].reset_index(drop=True)
    print(f"Đã lấy mẫu {len(df_test)} dòng từ test split.")
else:
    print(f"Dùng toàn bộ test split: {len(df_test)} dòng.")

df_test.head(3)

In [ ]:
# ---------------------------------------------------------------
# Phần 4: Khởi tạo LLMClient (Google Gemini)
# ---------------------------------------------------------------
gen_cfg  = cfg["generation"]
cost_cfg = cfg["cost"]

client = LLMClient(
    model=MODEL_NAME,
    api_key_env=cfg.get("api_key_env", "GOOGLE_API_KEY"),
    temperature=gen_cfg.get("temperature", 0.0),
    max_tokens=gen_cfg.get("max_tokens", 256),
    request_timeout=gen_cfg.get("request_timeout", 60),
    max_retries=gen_cfg.get("max_retries", 3),
    input_price_per_1k=cost_cfg.get("input_price_per_1k", 0.000075),
    output_price_per_1k=cost_cfg.get("output_price_per_1k", 0.0003),
    budget_cap_usd=cost_cfg.get("budget_cap_usd", 5.0),
)

# Few-shot examples (dùng khi mode="few_shot")
few_shot_examples: list = []
if PROMPTING_MODE == "few_shot":
    n = cfg["prompting"].get("num_few_shot_examples", 3)
    train_path = PROJECT_ROOT / "data" / "processed" / "train.csv"
    if train_path.exists() and cfg["prompting"].get("few_shot_selection") == "random":
        df_train = pd.read_csv(train_path)
        samples = df_train.sample(n=n, random_state=cfg["data"].get("sample_seed", 42))
        few_shot_examples = [
            {
                "text": row["text"],
                "label": label_map.get(int(row["label_id"]), "Normal"),
                "explanation": "Lấy từ training set.",
            }
            for _, row in samples.iterrows()
        ]
    else:
        few_shot_examples = _FEW_SHOT_EXAMPLES[:n]

print(f"✓ LLMClient khởi tạo thành công (model={MODEL_NAME})")
if PROMPTING_MODE == "few_shot":
    print(f"  Số ví dụ few-shot: {len(few_shot_examples)}")

In [ ]:
# ---------------------------------------------------------------
# Phần 5: Vòng lặp suy luận (Inference Loop)
# Kết quả được stream vào file JSONL ngay khi có — an toàn khi gián đoạn
# ---------------------------------------------------------------
from src.models.llm_client import LLMPrediction

jsonl_path = artifacts_dir / cfg["output"]["predictions_name"]
predictions: list[LLMPrediction] = []
parse_errors = 0

with open(jsonl_path, "w", encoding="utf-8") as fout:
    for _, row in tqdm(df_test.iterrows(), total=len(df_test), desc=f"Gemini [{MODEL_NAME}]"):
        try:
            pred = client.classify(
                text=str(row["text"]),
                label_map=label_map,
                mode=PROMPTING_MODE,
                few_shot_examples=few_shot_examples,
            )
        except RuntimeError as exc:
            logger.error("Budget cap: %s", exc)
            break

        if pred.parse_error:
            parse_errors += 1

        rec = {
            "text": pred.text,
            "true_label_id": int(row["label_id"]),
            "predicted_label": pred.predicted_label,
            "predicted_label_id": pred.predicted_label_id,
            "confidence": pred.confidence,
            "explanation": pred.explanation,
            "prompt_tokens": pred.prompt_tokens,
            "completion_tokens": pred.completion_tokens,
            "latency_s": round(pred.latency_s, 3),
            "parse_error": pred.parse_error,
        }
        fout.write(json.dumps(rec, ensure_ascii=False) + "\n")
        predictions.append(pred)

print(f"\n✓ Suy luận xong. Tổng: {len(predictions)} | Parse lỗi: {parse_errors}")
print(f"  Dự đoán đã lưu tại: {jsonl_path}")